In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import numpy as np

In [7]:
def make_monthly(df):
    df = df.sort_index().dropna(how = 'all')
    pxm = df.resample("M").last()
    retm = pxm.pct_change()
    #to make signal matches monthly return so signal every month and the result they predict are at the same place
    fwdret = retm.shift(-1)
    return pxm, retm, fwdret

In [8]:
def compute_factor(df, pxm, retm, fwdret):
    #momentum Pt/Pt-12 - 1
    mom = pxm/pxm.shift(12) - 1
    #find low volatility
    retd = df.pct_change()
    vol60 = retd.rolling(60).std()
    lowvol = -vol60.resample("M").last()
    return mom, lowvol

In [9]:
#zscore 
def countzscore(lowvol,mom,alpha):
    def findzscore(ser):
        #delete lower 0.05 and upper 0.05 limit, or std will be really large, and std will change more than mean, 
        #so most normal data's score will be really small, then the effect of this line will be weakened a lot
        ser2 = ser.clip(lower = ser.quantile(0.05), upper = ser.quantile(0.95))
        if ser2.std() == 0:
            return ser2*0
        #to make different factor score on the same amount level
        ser = (ser2 - ser2.mean()) / ser2.std()
        return ser
    volzscore = lowvol.apply(findzscore, axis = 1)
    momzscore = mom.apply(findzscore,axis = 1)
    zscore = alpha * volzscore + (1 - alpha) * momzscore
    return zscore

In [10]:
def build_weights(zscore, N):
    w = pd.DataFrame(0.0, index = zscore.index, columns = zscore.columns)
    for t in zscore.index:
        s = zscore.loc[t].dropna()
        if len(s) <= N:
            continue
        top = s.nlargest(N).index
        w.loc[t, top] = 1/N
    return w

In [11]:
def backtest(w, fwdret, tc):
    # portfolio_rett =i∑ wt,i ⋅ rt,i
    portret = (w * fwdret).sum(axis = 1, min_count = 1)
    #net of transaction costs
    turnover = w.diff().abs().sum(axis = 1)
    net = portret - tc * turnover
    net2 = net.dropna()
    #g12=(NAVT**1/T)**12=NAVT**12/T  σannual≈σmonthly *sqr12  Sharpe= （ Rannual−Rf ）/σannual
    nav = (net2 + 1).cumprod()
    annret = nav.iloc[-1]**(12/len((nav.dropna())))-1
    annvol = net2.std()*(12**0.5)
    sharp = annret / annvol
    #Max drawdown
    runningmax = nav.cummax()
    dd = nav / runningmax - 1
    ddmax = dd.min()
    return turnover, net2, nav, annret, annvol, sharp, ddmax

In [17]:
def oosperiod(net):
    ooslen = 24
    #seperate oos month and out oos month
    allmonth = net.index
    #find split month then seperate
    splitmonth = allmonth[-ooslen]
    ismonth = allmonth[allmonth < splitmonth]
    oosmonth = allmonth[allmonth >= splitmonth]
    return splitmonth, ismonth, oosmonth

    
def findsignal(df, splitmonth, tc):    
    #why not choose many signal groups?
    # Avoid data-snooping / selection bias: with many signal groups/parameter combos,
    # the best in-sample Sharpe is likely inflated by luck, hurting OOS generalization.
    #need to check sharp turnover(if it is high, it will be more sensitive to tc change), annret
    # qustion: still not so sure how to choose the right number of signal groups
    pxm, retm, fwdret = make_monthly(df.loc[df.index < splitmonth])
    mom, lowvol = compute_factor(df.loc[df.index < splitmonth], pxm, retm, fwdret)
    alpha = [0.25, 0.5, 0.75]
    N = [5, 10, 15]
    bestsharp = -np.inf
    alphamax = alpha[0]
    Nmax = N[0]
    for i in alpha:
        for j in N:
            zscore = countzscore(lowvol,mom,i)
            w = build_weights(zscore, j)
            turnover, net, nav, annret, annvol, sharp, ddmax = backtest(w, fwdret, tc)
            if sharp > bestsharp:
                alphamax =i
                Nmax = j
                bestsharp = sharp
    #backtest
    pxm, retm, fwdret = make_monthly(df)
    mom, lowvol = compute_factor(df, pxm, retm, fwdret)
    zscore = countzscore(lowvol,mom,alphamax)
    w = build_weights(zscore, Nmax)
    turnover, net, nav, annret, annvol, sharp, ddmax = backtest(w, fwdret, tc)
    netoos = net.loc[net.index >= splitmonth]
    netis = net.loc[net.index < splitmonth]
    netfull = net
    netoos = netoos.dropna()
    netis  = netis.dropna()
    nav = (netoos + 1).cumprod()
    annret = nav.iloc[-1]**(12/len((nav.dropna())))-1
    annvol = netoos.std()*(12**0.5)
    sharp = annret / annvol

    dd = nav / nav.cummax() - 1
    ddmax_oos = dd.min()
    return {
        "alpha_best": alphamax,
        "N_best": Nmax,
        "is_best_sharpe": bestsharp,
        "net_full": netfull,
        "net_is": netis,
        "net_oos": netoos,
        "nav_oos": nav,
        "annret_oos": annret,
        "annvol_oos": annvol,
        "sharpe_oos": sharp,
        "ddmax_oos": ddmax_oos,
    }

In [20]:
def main():
    tickers = ["AAPL","MSFT","NVDA","ADBE","GOOGL","META","NFLX","AMZN","TSLA","HD",
               "MCD","PG","KO","COST","JPM","BAC","GS","BRK-B","V","JNJ",
               "UNH","ABBV","CAT","HON","RTX","XOM","CVX","LIN","AMT","NEE"]
    data = yf.download(tickers, start="2015-01-01", auto_adjust=False)
    adj = data["Adj Close"].dropna(how="all")
    df = adj.copy()

    tc = 0.001

    pxm, retm, fwdret = make_monthly(df)
    mom, lowvol = compute_factor(df, pxm, retm, fwdret)

    alpha = 0.5
    N = 8
    zscore = countzscore(lowvol, mom, alpha)
    w = build_weights(zscore, N)
    turnover, net, nav, annret, annvol, sharp, ddmax = backtest(w, fwdret, tc)

    splitmonth, ismonth, oosmonth = oosperiod(net)

    netoos = net.loc[oosmonth].dropna()

    nav_oos = (netoos + 1).cumprod()
    annret_oos = nav_oos.iloc[-1]**(12/len(netoos)) - 1
    annvol_oos = netoos.std() * (12**0.5)
    sharpe_oos = annret_oos / annvol_oos if annvol_oos != 0 else np.nan
    dd_oos = nav_oos / nav_oos.cummax() - 1
    ddmax_oos = dd_oos.min()

    print("splitmonth:", splitmonth)
    print("FULL sharpe:", sharp, "FULL ddmax:", ddmax)
    print("OOS sharpe:", sharpe_oos, "OOS ddmax:", ddmax_oos)
    print("annret_oos", annret_oos)
    print("annvol_oos", annvol_oos)

In [21]:
main()

[*********************100%***********************]  30 of 30 completed
/var/folders/22/n4rqdwx117s71799081xqv3m0000gn/T/ipykernel_2079/4158309569.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  pxm = df.resample("M").last()
/var/folders/22/n4rqdwx117s71799081xqv3m0000gn/T/ipykernel_2079/2387580661.py:7: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  lowvol = -vol60.resample("M").last()


splitmonth: 2024-02-29 00:00:00
FULL sharpe: 0.9726088322417263 FULL ddmax: -0.15358802631450774
OOS sharpe: 1.0695107609641832 OOS ddmax: -0.07663740836533028
annret_oos 0.13506811018346232
annvol_oos 0.12628962242670283
